# MYR SupplyLens V2.1.1 — Runtime-Safe Google Colab Notebook

This is a full rewrite of the previous V2.1.1 setup notebook.

**Important safety change:** this notebook NEVER calls a Colab restart API, shuts down the IPython kernel, sends a kill signal, or force-reinstalls NumPy/SciPy/TensorFlow. It preserves the scientific package versions already present in your Colab runtime and stops safely if it detects an unexpected change.

If your previous Colab runtime disconnected, connect a fresh standard CPU/GPU runtime before running this notebook. Files saved in Google Drive are separate from the temporary runtime.

V2.1 forecasting/evaluation logic is unchanged. Existing Drive data, V1/V2/V2.1 artifacts and SQLite records are preserved.

In [1]:
# CELL 1 — Mount Google Drive and locate the patch archive.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, zipfile, shutil
from datetime import datetime
from pathlib import PurePosixPath

MY_DRIVE = Path('/content/drive/MyDrive')
PROJECT_DIR = MY_DRIVE / 'MYR_SupplyLens'
PATCH_ZIP = MY_DRIVE / 'MYR_SupplyLens_V2_1_1_Patch.zip'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if not PATCH_ZIP.is_file():
    raise FileNotFoundError(
        f'Patch archive not found: {PATCH_ZIP}\n'
        'Upload MYR_SupplyLens_V2_1_1_Patch.zip to the root of My Drive.'
    )

print('Project directory:', PROJECT_DIR)
print('Patch archive:', PATCH_ZIP)
print('Python executable:', sys.executable)

Mounted at /content/drive
Project directory: /content/drive/MyDrive/MYR_SupplyLens
Patch archive: /content/drive/MyDrive/MYR_SupplyLens_V2_1_1_Patch.zip
Python executable: /usr/bin/python3


In [2]:
# CELL 2 — Safely overlay the V2.1.1 source patch.
# Only source/code/documentation paths are allowed. data/ and artifacts/ are excluded.
ALLOWED_TOP_LEVEL_DIRS = {'src', 'pages', 'scripts', 'tests'}
ALLOWED_ROOT_FILES = {
    '.gitignore', 'README.md', 'README_V2_UPGRADE.md',
    'requirements-colab.txt', 'requirements-local.txt', 'app.py',
    'PATCH_MANIFEST.txt'
}

with zipfile.ZipFile(PATCH_ZIP, 'r') as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'Patch ZIP integrity check failed at: {bad_member}')

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_dir = PROJECT_DIR / 'backups' / f'source_backup_v2_1_1_{stamp}'
    written, backed_up, unchanged, skipped = [], [], [], []

    for info in archive.infolist():
        if info.is_dir():
            continue
        rel = PurePosixPath(info.filename)
        if rel.is_absolute() or '..' in rel.parts:
            skipped.append(info.filename)
            continue
        allowed = (rel.parts[0] in ALLOWED_TOP_LEVEL_DIRS) or (
            len(rel.parts) == 1 and rel.name in ALLOWED_ROOT_FILES
        )
        if not allowed:
            skipped.append(info.filename)
            continue

        payload = archive.read(info.filename)
        target = PROJECT_DIR.joinpath(*rel.parts)
        target.parent.mkdir(parents=True, exist_ok=True)

        if target.exists() and target.read_bytes() == payload:
            unchanged.append(str(rel))
            continue

        if target.exists():
            backup_target = backup_dir.joinpath(*rel.parts)
            backup_target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, backup_target)
            backed_up.append(str(rel))

        target.write_bytes(payload)
        written.append(str(rel))

print(f'Files updated: {len(written)}')
print(f'Existing changed files backed up: {len(backed_up)}')
print(f'Identical files skipped: {len(unchanged)}')
print(f'Unapproved archive entries skipped: {len(skipped)}')
print('Backup directory:', backup_dir if backed_up else 'No changed files needed backup')
print('Project root:', PROJECT_DIR)

Files updated: 1
Existing changed files backed up: 1
Identical files skipped: 29
Unapproved archive entries skipped: 0
Backup directory: /content/drive/MyDrive/MYR_SupplyLens/backups/source_backup_v2_1_1_20261114_082022
Project root: /content/drive/MyDrive/MYR_SupplyLens


## Dependency policy — preserve the Colab runtime

The earlier notebook pinned/downgraded core packages and then tried to restart Colab through an unavailable API. This notebook does neither.

It records the exact versions already installed for NumPy, pandas, SciPy, scikit-learn, statsmodels, TensorFlow and their key binary dependencies. Pip then installs the project requirements under a temporary constraints file containing those exact versions. If pip cannot satisfy the project without changing the Colab scientific stack, it fails safely rather than replacing those packages.

**There is no automatic restart.** If an import smoke test later reports stale modules, use Colab's menu manually: **Runtime → Restart session**, then rerun the notebook from the first cell. Never use process-kill commands.

In [3]:
# CELL 3 — Install only compatible project requirements; preserve core package versions.
import importlib.metadata as metadata
import subprocess

# These packages are expected in a standard Colab CPU/GPU runtime.
REQUIRED_COLAB_CORE = [
    'numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels', 'tensorflow'
]
# These are pinned to their already-installed versions when present because they
# are part of the compiled/scientific stack or TensorFlow runtime.
LOCKED_DISTRIBUTIONS = [
    'numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels',
    'tensorflow', 'keras', 'h5py', 'protobuf', 'ml-dtypes'
]

missing_core = []
core_before = {}
for dist in REQUIRED_COLAB_CORE:
    try:
        core_before[dist] = metadata.version(dist)
    except metadata.PackageNotFoundError:
        missing_core.append(dist)

if missing_core:
    raise RuntimeError(
        'This runtime is missing core packages: ' + ', '.join(missing_core) + '.\n'
        'Do not try to repair these by force-installing NumPy or TensorFlow. '
        'In Colab, select a standard CPU or GPU runtime (not TPU), reconnect, '
        'then rerun this notebook.'
    )

# Detect stale modules before pip is allowed to do anything. This can happen if
# packages were replaced earlier in the same live kernel. Stop safely and ask
# for a manual UI restart; never terminate the kernel from notebook code.
MODULE_TO_DIST = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
    'sklearn': 'scikit-learn', 'statsmodels': 'statsmodels',
    'tensorflow': 'tensorflow',
}
stale_imports = []
for module_name, dist_name in MODULE_TO_DIST.items():
    loaded_module = sys.modules.get(module_name)
    if loaded_module is None:
        continue
    loaded_version = getattr(loaded_module, '__version__', None)
    installed_version = metadata.version(dist_name)
    if loaded_version and loaded_version != installed_version:
        stale_imports.append((module_name, loaded_version, installed_version))

if stale_imports:
    raise RuntimeError(
        'Stale in-memory scientific modules detected: ' + repr(stale_imports) + '.\n'
        'No packages were installed or changed. In Colab, use Runtime → Restart session manually, '
        'then rerun this notebook from Cell 1. No kernel-kill or restart command is run by this notebook.'
    )

constraints_path = Path('/content/myr_supplylens_colab_constraints.txt')
constraint_lines = []
locked_before = {}
for dist in LOCKED_DISTRIBUTIONS:
    try:
        installed = metadata.version(dist)
        locked_before[dist] = installed
        constraint_lines.append(f'{dist}=={installed}')
    except metadata.PackageNotFoundError:
        # Optional distribution not present; do not install it just for pinning.
        pass

constraints_path.write_text('\n'.join(constraint_lines) + '\n', encoding='utf-8')
print('Existing core versions (will be preserved):')
for name, version in core_before.items():
    print(f'  {name}: {version}')
print('Constraints file:', constraints_path)

requirements_file = PROJECT_DIR / 'requirements-colab.txt'
if not requirements_file.is_file():
    raise FileNotFoundError(f'Missing requirements file: {requirements_file}')

command = [
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check',
    '--upgrade-strategy', 'only-if-needed',
    '--constraint', str(constraints_path),
    '--requirement', str(requirements_file),
]
print('Installing project requirements without replacing the locked scientific stack...')
subprocess.run(command, check=True)

core_after = {name: metadata.version(name) for name in core_before}
changed_core = {
    name: (core_before[name], core_after[name])
    for name in core_before if core_before[name] != core_after[name]
}
if changed_core:
    raise RuntimeError(
        'Safety check stopped: one or more core packages changed unexpectedly: '
        + repr(changed_core) + '.\n'
        'No automatic restart was attempted. Use Runtime → Restart session manually, '
        'then rerun from the first cell.'
    )

print('\nDependency install completed. Core package versions are unchanged.')
print('No runtime restart was attempted.')

Existing core versions (will be preserved):
  numpy: 2.4.6
  pandas: 2.2.3
  scipy: 1.16.3
  scikit-learn: 1.6.1
  statsmodels: 0.15.0
  tensorflow: 2.21.0
Constraints file: /content/myr_supplylens_colab_constraints.txt
Installing project requirements without replacing the locked scientific stack...

Dependency install completed. Core package versions are unchanged.
No runtime restart was attempted.


In [4]:
# CELL 4 — Scientific stack import smoke test.
# This is deliberately after dependency installation.
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels
import tensorflow as tf
from scipy.sparse import csr_matrix
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.arima.model import ARIMA

print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('statsmodels:', statsmodels.__version__)
print('TensorFlow:', tf.__version__)
print('SciPy sparse smoke test:', csr_matrix([[1, 0], [0, 1]]).shape)
print('scikit-learn smoke test:', MinMaxScaler().fit_transform([[1.0], [2.0]]).shape)
print('ARIMA class:', ARIMA.__name__)

gpus = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpus if gpus else 'None — CPU mode')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print('GPU memory-growth setting skipped:', exc)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print('\nMYR SupplyLens V2.1.1 runtime preflight: PASSED')

Python: 3.13.16
NumPy: 2.4.6
Pandas: 2.2.3
SciPy: 1.16.3
scikit-learn: 1.6.1
statsmodels: 0.15.0
TensorFlow: 2.21.0
SciPy sparse smoke test: (2, 2)
scikit-learn smoke test: (2, 1)
ARIMA class: ARIMA
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

MYR SupplyLens V2.1.1 runtime preflight: PASSED


## Run source tests and inspect exchange-rate data

In [5]:

# CELL 5 — V2.1.1 controlled unit test execution
# Runs only the four official V2.1.1 test modules.
# Prevents unrelated legacy tests and third-party pytest plugins
# from interfering with test collection.

from pathlib import Path
import os
import sys
import subprocess

PROJECT_DIR = Path(PROJECT_DIR).resolve()

TEST_FILES = [
    PROJECT_DIR / "tests" / "test_business.py",
    PROJECT_DIR / "tests" / "test_features.py",
    PROJECT_DIR / "tests" / "test_evaluation.py",
    PROJECT_DIR / "tests" / "test_forecasting.py",
]

# 1. Verify project and test files
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory missing: {PROJECT_DIR}")

missing = [str(p) for p in TEST_FILES if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Required V2.1.1 test files are missing:\n"
        + "\n".join(missing)
        + "\nReapply the source patch before running tests."
    )

# 2. Configure an isolated pytest execution environment
env = os.environ.copy()

env["PYTHONPATH"] = (
    str(PROJECT_DIR)
    + os.pathsep
    + env.get("PYTHONPATH", "")
)

# Avoid unrelated globally installed pytest plugins.
env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"

# 3. Run only the official V2.1.1 tests
command = [
    sys.executable,
    "-m",
    "pytest",
    "-q",
    "-ra",
    "--tb=short",
    "-p",
    "no:cacheprovider",
    *[str(p.relative_to(PROJECT_DIR)) for p in TEST_FILES],
]

print("MYR SupplyLens V2.1.1 — Unit Tests")
print("=" * 55)
print("Project:", PROJECT_DIR)
print("Python:", sys.executable)
print("Test modules:", len(TEST_FILES))
print("=" * 55)

result = subprocess.run(
    command,
    cwd=str(PROJECT_DIR),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# 4. Always display the actual pytest output
print(result.stdout or "Pytest returned no output.")

if result.returncode != 0:
    raise RuntimeError(
        f"V2.1.1 tests failed with exit code {result.returncode}. "
        "See the pytest output immediately above."
    )

print("\n" + "=" * 55)
print("SUCCESS: All V2.1.1 unit tests passed.")
print("=" * 55)


MYR SupplyLens V2.1.1 — Unit Tests
Project: /content/drive/MyDrive/MYR_SupplyLens
Python: /usr/bin/python3
Test modules: 4
...........                                                              [100%]
11 passed in 7.89s


SUCCESS: All V2.1.1 unit tests passed.


In [10]:

# ============================================================
# MYR SUPPLYLENS V2.1.2 — NUMERICAL STABILITY PATCH
# In-place source repair. No installations or runtime restarts.
# ============================================================

from pathlib import Path
import re
import sys
import os
import py_compile
import subprocess

PROJECT_DIR = Path(PROJECT_DIR).resolve()
EVAL_FILE = PROJECT_DIR / "src" / "supplylens" / "evaluation.py"

if not EVAL_FILE.exists():
    raise FileNotFoundError(EVAL_FILE)

source = EVAL_FILE.read_text(encoding="utf-8")

# Preserve original source once.
backup = EVAL_FILE.with_name("evaluation.py.v2_1_1_backup")

if not backup.exists():
    backup.write_text(source, encoding="utf-8")
    print("Original evaluation.py backed up.")

def replace_function(src, name, replacement):
    pattern = rf"(?ms)^def {re.escape(name)}\(.*?(?=^def |\Z)"
    updated, count = re.subn(
        pattern, replacement.strip() + "\n\n", src, count=1
    )
    if count != 1:
        raise RuntimeError(f"Cannot safely replace function: {name}")
    return updated

def replace_block(src, old, new, description):
    if new in src:
        print("Already applied:", description)
        return src
    if old not in src:
        raise RuntimeError(
            "Cannot locate source block: " + description
            + ". No further modifications were made."
        )
    print("Applying:", description)
    return src.replace(old, new, 1)


# ------------------------------------------------------------
# A. SAFE NUMERICAL HELPERS
# ------------------------------------------------------------

helpers = r'''
def _forecast_log_move_limit(log_history, horizon):
    """Volatility-aware guardrail. Reject extreme forecasts; never clip."""
    history = np.asarray(log_history, dtype=np.float64).reshape(-1)

    if len(history) < 2 or not np.isfinite(history).all():
        return None

    returns = np.diff(history)
    if not np.isfinite(returns).all():
        return None

    median = float(np.median(returns))
    mad = float(np.median(np.abs(returns - median)))
    robust_sigma = 1.4826 * mad

    if not np.isfinite(robust_sigma):
        return None

    limit = 10.0 * robust_sigma * np.sqrt(max(1, int(horizon)))
    return float(np.clip(limit, 0.10, 0.50))


def _safe_rate_path_from_log_levels(log_levels, base_rate, log_history, horizon):
    levels = np.asarray(log_levels, dtype=np.float64).reshape(-1)
    base_rate = float(base_rate)
    horizon = int(horizon)

    if (
        horizon < 1
        or len(levels) != horizon
        or not np.isfinite(levels).all()
        or not np.isfinite(base_rate)
        or base_rate <= 0
    ):
        return None

    base_log = float(np.log(base_rate))
    limit = _forecast_log_move_limit(log_history, horizon)

    if limit is None or np.any(np.abs(levels - base_log) > limit):
        return None

    max_log = np.log(np.finfo(np.float64).max) - 2.0
    min_log = np.log(np.finfo(np.float64).tiny) + 2.0

    if np.any(levels > max_log) or np.any(levels < min_log):
        return None

    with np.errstate(over="ignore", under="ignore", invalid="ignore"):
        rates = np.exp(levels)

    if not np.isfinite(rates).all() or np.any(rates <= 0):
        return None

    return rates


def _safe_rate_path_from_log_returns(base_rate, log_returns, log_history, horizon):
    log_returns = np.asarray(log_returns, dtype=np.float64).reshape(-1)
    base_rate = float(base_rate)
    horizon = int(horizon)

    if (
        len(log_returns) != horizon
        or not np.isfinite(log_returns).all()
        or not np.isfinite(base_rate)
        or base_rate <= 0
    ):
        return None

    with np.errstate(over="ignore", invalid="ignore"):
        levels = np.log(base_rate) + np.cumsum(log_returns)

    if not np.isfinite(levels).all():
        return None

    return _safe_rate_path_from_log_levels(
        levels, base_rate, log_history, horizon
    )


def _stable_rmse(errors):
    """Calculate RMSE without directly squaring potentially large values."""
    errors = np.asarray(errors, dtype=np.float64)

    if len(errors) == 0 or not np.isfinite(errors).all():
        return np.nan

    scale = float(np.max(np.abs(errors)))

    if scale == 0.0:
        return 0.0

    return float(
        scale * np.sqrt(np.mean(np.square(errors / scale)))
    )
'''

if "def _forecast_log_move_limit(" not in source:
    source = source.replace(
        "def _inverse_return_predictions(",
        helpers + "\n\n\ndef _inverse_return_predictions(",
        1
    )


# ------------------------------------------------------------
# B. REPLACE CNN-LSTM PREDICTION VALIDATION
# ------------------------------------------------------------

new_inverse = r'''
def _inverse_return_predictions(model, X_scaled, scaler_y):
    pred_scaled = np.asarray(
        model.predict(X_scaled, verbose=0), dtype=np.float64
    )

    if pred_scaled.ndim != 2:
        raise ValueError(
            f"CNN-LSTM must return a 2D array; got {pred_scaled.shape}."
        )

    if not np.isfinite(pred_scaled).all():
        bad = int((~np.isfinite(pred_scaled)).sum())
        raise ValueError(
            f"CNN-LSTM produced {bad} non-finite scaled predictions."
        )

    pred_returns = scaler_y.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).reshape(pred_scaled.shape)

    if not np.isfinite(pred_returns).all():
        bad = int((~np.isfinite(pred_returns)).sum())
        raise ValueError(
            f"CNN-LSTM produced {bad} non-finite log-return predictions."
        )

    return pred_returns
'''

source = replace_function(
    source, "_inverse_return_predictions", new_inverse
)


# ------------------------------------------------------------
# C. REPLACE ARIMA FORECASTING
# ------------------------------------------------------------

new_arima = r'''
def _fit_arima_forecast(log_history, horizon):
    """Fit ARIMA(1,1,1), rejecting unstable or implausible forecasts."""
    try:
        from statsmodels.tsa.arima.model import ARIMA

        history = np.asarray(
            log_history, dtype=np.float64
        ).reshape(-1)

        horizon = int(horizon)

        if (
            horizon < 1
            or len(history) < 30
            or not np.isfinite(history).all()
        ):
            return None

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            fitted = ARIMA(
                history,
                order=(1, 1, 1),
                trend="n",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit()

            forecast_log = np.asarray(
                fitted.forecast(steps=horizon),
                dtype=np.float64
            ).reshape(-1)

        if len(forecast_log) != horizon:
            return None

        with np.errstate(over="ignore", under="ignore", invalid="ignore"):
            base_rate = float(np.exp(history[-1]))

        if not np.isfinite(base_rate) or base_rate <= 0:
            return None

        return _safe_rate_path_from_log_levels(
            forecast_log,
            base_rate,
            history,
            horizon,
        )

    except Exception:
        # ARIMA is optional. Its failure must not terminate other models.
        return None
'''

source = replace_function(source, "_fit_arima_forecast", new_arima)


# ------------------------------------------------------------
# D. REPLACE BENCHMARK FORECAST PATH GENERATION
# ------------------------------------------------------------

new_benchmark = r'''
def benchmark_forecast_paths(
    log_history, base_rate, horizon,
    cnn_path=None, include_arima=True
):
    """Return only valid, positive and finite forecasting paths."""

    log_history = np.asarray(
        log_history, dtype=np.float64
    ).reshape(-1)

    base_rate = float(base_rate)
    horizon = int(horizon)

    if horizon < 1:
        raise ValueError("Forecast horizon must be positive.")

    if len(log_history) < 2 or not np.isfinite(log_history).all():
        raise ValueError("Historical log-rate observations are invalid.")

    if not np.isfinite(base_rate) or base_rate <= 0:
        raise ValueError("Base exchange rate must be finite and positive.")

    returns = np.diff(log_history)

    if not np.isfinite(returns).all():
        raise ValueError("Historical log returns contain invalid values.")

    drift_252 = float(np.mean(returns[-252:]))
    drift_20 = float(np.mean(returns[-20:]))

    paths = {
        "Random walk": np.full(
            horizon, base_rate, dtype=np.float64
        )
    }

    historical_path = _safe_rate_path_from_log_returns(
        base_rate,
        np.full(horizon, drift_252),
        log_history,
        horizon,
    )

    recent_path = _safe_rate_path_from_log_returns(
        base_rate,
        np.full(horizon, drift_20),
        log_history,
        horizon,
    )

    if historical_path is not None:
        paths["Historical drift (252)"] = historical_path

    if recent_path is not None:
        paths["Recent drift (20)"] = recent_path

    if cnn_path is not None:
        candidate = np.asarray(
            cnn_path, dtype=np.float64
        ).reshape(-1)

        if (
            len(candidate) == horizon
            and np.isfinite(candidate).all()
            and np.all(candidate > 0)
        ):
            candidate_log = np.log(candidate)
            base_log = np.log(base_rate)
            limit = _forecast_log_move_limit(log_history, horizon)

            if (
                limit is not None
                and np.all(np.abs(candidate_log - base_log) <= limit)
            ):
                paths["CNN-LSTM"] = candidate

    if include_arima:
        arima_path = _fit_arima_forecast(
            log_history[-(ARIMA_MAX_HISTORY + 1):],
            horizon,
        )

        if arima_path is not None:
            paths["ARIMA(1,1,1)"] = arima_path

    return paths
'''

source = replace_function(
    source, "benchmark_forecast_paths", new_benchmark
)


# ------------------------------------------------------------
# E. REPLACE METRIC CALCULATION
# ------------------------------------------------------------

new_metric = r'''
def _metric_row(currency, horizon, model_name, actual, predicted, base):
    actual = np.asarray(actual, dtype=np.float64).reshape(-1)
    predicted = np.asarray(predicted, dtype=np.float64).reshape(-1)
    base = np.asarray(base, dtype=np.float64).reshape(-1)

    if not (len(actual) == len(predicted) == len(base)):
        raise ValueError("Metric inputs must have equal lengths.")

    valid = (
        np.isfinite(actual)
        & np.isfinite(predicted)
        & np.isfinite(base)
        & (actual > 0)
        & (predicted > 0)
        & (base > 0)
    )

    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        errors_all = np.abs(actual - predicted)
        percentage_errors_all = errors_all / actual

    valid &= (
        np.isfinite(errors_all)
        & np.isfinite(percentage_errors_all)
    )

    actual = actual[valid]
    predicted = predicted[valid]
    base = base[valid]
    absolute_errors = errors_all[valid]
    percentage_errors = percentage_errors_all[valid]

    if len(actual) == 0:
        raise ValueError(
            f"No valid forecast pairs for {model_name}, "
            f"{currency}, horizon {horizon}."
        )

    mae = float(np.mean(absolute_errors))
    rmse = _stable_rmse(absolute_errors)
    mape = float(np.mean(percentage_errors) * 100.0)

    if model_name == "Random walk":
        direction = np.nan
        direction_status = "N/A — predicts no directional change"
    else:
        direction = float(
            np.mean(
                np.sign(actual - base)
                == np.sign(predicted - base)
            ) * 100.0
        )
        direction_status = "Calculated"

    return {
        "currency": currency,
        "horizon": int(horizon),
        "model": model_name,
        "n_origins": int(len(actual)),
        "mae": mae,
        "rmse": rmse,
        "mape_pct": mape,
        "absolute_error_std": (
            float(np.std(absolute_errors, ddof=1))
            if len(absolute_errors) > 1 else np.nan
        ),
        "directional_accuracy_pct": direction,
        "directional_status": direction_status,
    }
'''

source = replace_function(source, "_metric_row", new_metric)


# ------------------------------------------------------------
# F. PATCH WALK-FORWARD CNN PATH AND REJECTION REPORTING
# ------------------------------------------------------------

source = replace_block(
    source,
    '''    detail_rows = []
    for h in horizons:
''',
    '''    detail_rows = []
    rejection_counts = {}
    for h in horizons:
''',
    "Walk-forward rejection counter"
)

source = replace_block(
    source,
    '''            cnn_path = base_rate * np.exp(np.cumsum(pred_returns[row_idx]))
            history_start = max(0, origin - ARIMA_MAX_HISTORY)
            log_history = np.log(aligned_rates[history_start:origin + 1])
            paths = benchmark_forecast_paths(
''',
    '''            history_start = max(0, origin - ARIMA_MAX_HISTORY)
            history_rates = np.asarray(
                aligned_rates[history_start:origin + 1],
                dtype=np.float64
            )

            if (
                len(history_rates) < 2
                or not np.isfinite(history_rates).all()
                or np.any(history_rates <= 0)
            ):
                continue

            log_history = np.log(history_rates)

            cnn_path = _safe_rate_path_from_log_returns(
                base_rate,
                pred_returns[row_idx],
                log_history,
                pred_returns.shape[1],
            )

            paths = benchmark_forecast_paths(
''',
    "Walk-forward CNN-LSTM safe conversion"
)

source = replace_block(
    source,
    '''            if arima_path is not None:
                paths["ARIMA(1,1,1)"] = arima_path

            origin_date = pd.Timestamp(aligned_dates[origin]).date().isoformat()
''',
    '''            if arima_path is not None:
                paths["ARIMA(1,1,1)"] = arima_path

            expected_methods = [
                "CNN-LSTM",
                "Historical drift (252)",
                "Recent drift (20)",
            ]

            if include_arima:
                expected_methods.append("ARIMA(1,1,1)")

            for candidate_name in expected_methods:
                if candidate_name not in paths:
                    key = (int(h), candidate_name)
                    rejection_counts[key] = (
                        rejection_counts.get(key, 0) + 1
                    )

            origin_date = pd.Timestamp(aligned_dates[origin]).date().isoformat()
''',
    "Walk-forward rejected-method tracking"
)

source = replace_block(
    source,
    '''        metric_rows.append(_metric_row(
            currency, horizon, model_name,
            group["actual_rate"].to_numpy(),
            group["forecast_rate"].to_numpy(),
            group["base_rate"].to_numpy(),
        ))
    metrics = pd.DataFrame(metric_rows)
''',
    '''        metric_row = _metric_row(
            currency, horizon, model_name,
            group["actual_rate"].to_numpy(),
            group["forecast_rate"].to_numpy(),
            group["base_rate"].to_numpy(),
        )

        metric_row["n_rejected"] = int(
            rejection_counts.get((int(horizon), model_name), 0)
        )
        metric_row["n_attempted"] = (
            metric_row["n_origins"] + metric_row["n_rejected"]
        )

        metric_rows.append(metric_row)

    # Retain candidates that were rejected at every evaluation origin.
    candidate_methods = [
        "CNN-LSTM",
        "Historical drift (252)",
        "Recent drift (20)",
    ]

    if include_arima:
        candidate_methods.append("ARIMA(1,1,1)")

    present = {
        (int(row["horizon"]), str(row["model"]))
        for row in metric_rows
    }

    for h in horizons:
        for method in candidate_methods:
            key = (int(h), method)

            if key in present or rejection_counts.get(key, 0) == 0:
                continue

            rejected = int(rejection_counts[key])

            metric_rows.append({
                "currency": currency,
                "horizon": int(h),
                "model": method,
                "n_origins": 0,
                "mae": np.nan,
                "rmse": np.nan,
                "mape_pct": np.nan,
                "absolute_error_std": np.nan,
                "directional_accuracy_pct": np.nan,
                "directional_status": "No valid forecasts; candidate rejected",
                "n_rejected": rejected,
                "n_attempted": rejected,
            })

    metrics = pd.DataFrame(metric_rows)
''',
    "Walk-forward rejection metrics"
)


# ------------------------------------------------------------
# G. PATCH CALIBRATION
# ------------------------------------------------------------

source = replace_block(
    source,
    '''    error_store = {}

    for origin in origins:
''',
    '''    error_store = {}
    rejected_store = {}

    for origin in origins:
''',
    "Calibration rejection counter"
)

source = replace_block(
    source,
    '''        cnn_path = base_rate * np.exp(np.cumsum(pred_returns[origin_to_row[origin]]))
        history_start = max(0, origin - ARIMA_MAX_HISTORY)
        log_history = np.log(aligned_rates[history_start:origin + 1])
        model_horizon = int(pred_returns.shape[1])
''',
    '''        history_start = max(0, origin - ARIMA_MAX_HISTORY)
        history_rates = np.asarray(
            aligned_rates[history_start:origin + 1],
            dtype=np.float64
        )

        if (
            len(history_rates) < 2
            or not np.isfinite(history_rates).all()
            or np.any(history_rates <= 0)
        ):
            continue

        log_history = np.log(history_rates)
        model_horizon = int(pred_returns.shape[1])

        cnn_path = _safe_rate_path_from_log_returns(
            base_rate,
            pred_returns[origin_to_row[origin]],
            log_history,
            model_horizon,
        )
''',
    "Calibration CNN-LSTM safe conversion"
)

source = replace_block(
    source,
    '''        for method, path in paths.items():
            errors = np.abs(actual_log - np.log(np.asarray(path, dtype=float)[:horizon]))
            for step, error in enumerate(errors, start=1):
                error_store.setdefault((method, step), []).append(float(error))

    rows = []
    methods = sorted({method for method, _ in error_store})
''',
    '''        expected_methods = [
            "CNN-LSTM",
            "Random walk",
            "Historical drift (252)",
            "Recent drift (20)",
        ]

        if include_arima:
            expected_methods.append("ARIMA(1,1,1)")

        for method in expected_methods:
            path = paths.get(method)

            valid_path = (
                path is not None
                and len(path) >= horizon
                and np.isfinite(
                    np.asarray(path[:horizon], dtype=np.float64)
                ).all()
                and np.all(
                    np.asarray(path[:horizon], dtype=np.float64) > 0
                )
            )

            if not valid_path:
                for step in range(1, horizon + 1):
                    key = (method, step)
                    rejected_store[key] = rejected_store.get(key, 0) + 1
                continue

            with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                errors = np.abs(
                    actual_log
                    - np.log(np.asarray(path[:horizon], dtype=np.float64))
                )

            for step, error in enumerate(errors, start=1):
                key = (method, step)

                if np.isfinite(error):
                    error_store.setdefault(key, []).append(float(error))
                else:
                    rejected_store[key] = rejected_store.get(key, 0) + 1

    rows = []
    methods = [
        "CNN-LSTM",
        "Random walk",
        "Historical drift (252)",
        "Recent drift (20)",
    ]

    if include_arima:
        methods.append("ARIMA(1,1,1)")
''',
    "Calibration invalid-path rejection"
)

source = replace_block(
    source,
    '''                "calibration_samples": count,
                "interval_available": bool(available),
''',
    '''                "calibration_samples": count,
                "rejected_forecasts": int(
                    rejected_store.get((method, step), 0)
                ),
                "interval_available": bool(available),
''',
    "Calibration rejection reporting"
)


# ------------------------------------------------------------
# H. SAVE AND COMPILE
# ------------------------------------------------------------

EVAL_FILE.write_text(source, encoding="utf-8")

py_compile.compile(str(EVAL_FILE), doraise=True)

print("\n" + "=" * 60)
print("V2.1.2 PATCH APPLIED SUCCESSFULLY")
print("=" * 60)
print("File:", EVAL_FILE)
print("Backup:", backup)
print("Compilation: PASSED")
print("Runtime restart: NOT REQUIRED")
print("Package installation: NOT REQUIRED")


# ------------------------------------------------------------
# I. RUN THE EXISTING TEST SUITE
# ------------------------------------------------------------

test_files = [
    "tests/test_business.py",
    "tests/test_features.py",
    "tests/test_evaluation.py",
    "tests/test_forecasting.py",
]

missing = [
    name for name in test_files
    if not (PROJECT_DIR / name).is_file()
]

if missing:
    print("\nSome test files were not found:", missing)
    print("Run your existing controlled test cell separately.")
else:
    env = os.environ.copy()
    env["PYTHONPATH"] = (
        str(PROJECT_DIR)
        + os.pathsep
        + env.get("PYTHONPATH", "")
    )
    env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"

    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "-ra", "--tb=short",
            "-p", "no:cacheprovider",
            *test_files,
        ],
        cwd=str(PROJECT_DIR),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print("\nTEST RESULTS")
    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"V2.1.2 tests failed: exit code {result.returncode}"
        )

    print("All existing V2.1.1 tests passed.")


Original evaluation.py backed up.
Applying: Walk-forward rejection counter
Applying: Walk-forward CNN-LSTM safe conversion
Applying: Walk-forward rejected-method tracking
Applying: Walk-forward rejection metrics
Applying: Calibration rejection counter
Applying: Calibration CNN-LSTM safe conversion
Applying: Calibration invalid-path rejection
Applying: Calibration rejection reporting

V2.1.2 PATCH APPLIED SUCCESSFULLY
File: /content/drive/MyDrive/MYR_SupplyLens/src/supplylens/evaluation.py
Backup: /content/drive/MyDrive/MYR_SupplyLens/src/supplylens/evaluation.py.v2_1_1_backup
Compilation: PASSED
Runtime restart: NOT REQUIRED
Package installation: NOT REQUIRED

TEST RESULTS
...........                                                              [100%]
11 passed in 1.83s

All existing V2.1.1 tests passed.


In [11]:
# CELL 6 — Inspect data availability and currency columns.
subprocess.run([sys.executable, 'scripts/inspect_data.py'], cwd=PROJECT_DIR, check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/inspect_data.py'], returncode=0)

## Train V2.1 model bundles (optional)

V2.1.1 is a dependency/runtime repair; the forecasting/evaluation logic and `_v2_1_` artifact naming remain unchanged. If you already have trained V2.1 bundles, you do not need to retrain solely because of this hotfix.

Set `RUN_TRAINING = True` to train. Start with USD, verify the output, then add CNY and IDR. Training runs sequentially and writes new V2.1 model files without overwriting V1/V2 bundles.

In [12]:

# MYR SupplyLens V2.1.1 — Training Hotfix
# Fix missing CALIBRATION_MIN_SAMPLES import
# Then rerun USD training with visible error output.

from pathlib import Path
import sys
import os
import subprocess
import py_compile

PROJECT_DIR = Path(PROJECT_DIR).resolve()
training_file = PROJECT_DIR / "src/supplylens/training.py"

if not training_file.exists():
    raise FileNotFoundError(f"Missing source file: {training_file}")

# --------------------------------------------------
# 1. Patch missing import
# --------------------------------------------------

source = training_file.read_text(encoding="utf-8")

start = source.index("from .config import (")
end = source.index(")", start)
import_block = source[start:end]

if "CALIBRATION_MIN_SAMPLES" not in import_block:
    needle = (
        "    WALK_FORWARD_SPACING, MIN_SELECTION_ORIGINS, "
        "CALIBRATION_ORIGIN_STRIDE,\n"
    )

    if needle not in source:
        raise RuntimeError(
            "Could not locate expected config import statement. "
            "Source structure differs from expected V2.1.1."
        )

    source = source.replace(
        needle,
        needle + "    CALIBRATION_MIN_SAMPLES,\n",
        1
    )

    training_file.write_text(source, encoding="utf-8")
    print("PATCH APPLIED: CALIBRATION_MIN_SAMPLES imported.")
else:
    print("Import already exists. No source modification needed.")

# --------------------------------------------------
# 2. Compile-check the patched module
# --------------------------------------------------

py_compile.compile(str(training_file), doraise=True)
print("Source compilation: PASSED")

# --------------------------------------------------
# 3. Run USD training with complete output
# --------------------------------------------------

command = [
    sys.executable,
    "scripts/train_model.py",
    "--currency", "USD",
    "--epochs", "40",
    "--batch-size", "32",
    "--calibration-stride", "5",
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print("\nStarting MYR SupplyLens USD training...")
print("Command:", " ".join(command))
print("=" * 65)

process = subprocess.Popen(
    command,
    cwd=str(PROJECT_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

print("\n" + "=" * 65)

if return_code != 0:
    raise RuntimeError(
        f"USD training failed with exit code {return_code}. "
        "The actual Python traceback is displayed above."
    )

print("SUCCESS: USD V2.1.1 training completed.")


Import already exists. No source modification needed.
Source compilation: PASSED

Starting MYR SupplyLens USD training...
Command: /usr/bin/python3 scripts/train_model.py --currency USD --epochs 40 --batch-size 32 --calibration-stride 5
W0000 00:00:1794646395.338378    9975 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Epoch 1/40

  1/133 ━━━━━━━━━━━━━━━━━━━━ 8:03 4s/step - loss: 1.7281 - mae: 2.1752
  7/133 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 2.1663 - mae: 2.6127 
 15/133 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.6682 - mae: 2.0418
 23/133 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.3342 - mae: 1.6462

## Inspect V2.1 evaluation and method-selection outputs

In [13]:
# CELL 8 — Display model metrics and validation-selected policy files.
import pandas as pd
from src.supplylens.config import METADATA_DIR, MODEL_VERSION

for currency in ['USD', 'CNY', 'IDR']:
    metrics_path = METADATA_DIR / f'metrics_{MODEL_VERSION}_{currency}.csv'
    policy_path = METADATA_DIR / f'selection_policy_{MODEL_VERSION}_{currency}.csv'
    if metrics_path.exists():
        print(f'\n{currency} — untouched final-test metrics')
        display(pd.read_csv(metrics_path))
    else:
        print(f'No V2.1 metrics yet for {currency}')
    if policy_path.exists():
        print(f'{currency} — validation-selected policy')
        display(pd.read_csv(policy_path))


USD — untouched final-test metrics


,currency,horizon,model,n_origins,mae,rmse,mape_pct,absolute_error_std,directional_accuracy_pct,directional_status,...,n_attempted,mae_skill_vs_random_walk_pct,mae_skill_ci_lower_pct,mae_skill_ci_upper_pct,rmse_skill_vs_random_walk_pct,rmse_skill_ci_lower_pct,rmse_skill_ci_upper_pct,skill_vs_random_walk_pct,skill_ci_lower_pct,skill_ci_upper_pct
0,USD,5,"ARIMA(1,1,1)",17,0.027935,0.047584,0.647471,0.039707,64.705882,Calculated,...,17,1.475109,-0.662830,4.491115,1.072718,0.173092,2.658873,1.475109,-0.662830,4.491115
1,USD,5,CNN-LSTM,17,0.028267,0.047971,0.654335,0.039951,52.941176,Calculated,...,17,0.302974,-0.969422,1.573616,0.267480,-1.210814,0.659237,0.302974,-0.969422,1.573616
2,USD,5,Historical drift (252),17,0.028533,0.047504,0.659973,0.039149,58.823529,Calculated,...,17,-0.633904,-14.090988,7.420990,1.238436,-5.750942,8.395433,-0.633904,-14.090988,7.420990
3,USD,5,Random walk,17,0.028353,0.048100,0.656738,0.040051,NaN,N/A — predicts no directional change,...,17,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,USD,5,Recent drift (20),17,0.032192,0.043234,0.741112,0.029747,58.823529,Calculated,...,17,-13.540191,-85.934478,14.970527,10.115921,-50.945070,30.034089,-13.540191,-85.934478,14.970527
5,USD,21,"ARIMA(1,1,1)",17,0.053901,0.064873,1.227567,0.037210,52.941176,Calculated,...,17,0.831829,-0.315124,2.436596,0.313053,-0.496754,1.391402,0.831829,-0.315124,2.436596
6,USD,21,CNN-LSTM,17,0.054251,0.065187,1.233400,0.037253,52.941176,Calculated,...,17,0.187710,-2.127972,2.708057,-0.169856,-2.087508,1.946858,0.187710,-2.127972,2.708057
7,USD,21,Historical drift (252),17,0.054386,0.063343,1.235769,0.033472,58.823529,Calculated,...,17,-0.060570,-23.276126,16.138696,2.664424,-19.455582,18.531320,-0.060570,-23.276126,16.138696
8,USD,21,Random walk,17,0.054353,0.065076,1.237173,0.036888,NaN,N/A — predicts no directional change,...,17,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,USD,21,Recent drift (20),17,0.094298,0.122192,2.132625,0.080101,58.823529,Calculated,...,17,-73.492378,-149.433408,-12.636378,-87.767090,-168.071387,-23.459735,-73.492378,-149.433408,-12.636378


USD — validation-selected policy


,currency,horizon,selected_model,selection_metric,selection_mae,selection_rmse,n_selection_origins,eligible_candidate_count,selection_fallback,selection_reason
0,USD,5,Random walk,selection-partition MAE,0.020489,0.028335,134,5,False,Lowest selection-partition MAE; RMSE and error...
1,USD,21,Random walk,selection-partition MAE,0.049766,0.060239,32,5,False,Lowest selection-partition MAE; RMSE and error...
2,USD,42,"ARIMA(1,1,1)",selection-partition MAE,0.080080,0.094045,14,5,False,Lowest selection-partition MAE; RMSE and error...
3,USD,63,Random walk,selection-partition MAE,0.096500,0.108068,11,5,False,Lowest selection-partition MAE; RMSE and error...


No V2.1 metrics yet for CNY
No V2.1 metrics yet for IDR


## Launch Streamlit (optional)

Run this after the runtime preflight and tests. If the app is already running in another cell, do not launch a duplicate server.

In [17]:

# ============================================================
# MYR SUPPLYLENS V2.1.1
# STREAMLIT COLAB WEBSOCKET FIX
#
# Fixes:
# - Colab reverse-proxy Origin rejection
# - Streamlit WebSocket handshake failure
#
# Preserves:
# - Google Drive data
# - Trained CNN-LSTM models
# - Existing project files
# - Current Colab runtime
# ============================================================

import sys
import time
import subprocess
import urllib.request
from pathlib import Path
from google.colab import output

PROJECT_DIR = Path(PROJECT_DIR)
PORT = 8501

print("=" * 60)
print("MYR SUPPLYLENS — STREAMLIT REPAIR")
print("=" * 60)

# ------------------------------------------------------------
# 1. Gracefully stop the existing Streamlit process
# ------------------------------------------------------------

old_server = globals().get("server")

if old_server is not None and old_server.poll() is None:
    print("\nStopping existing Streamlit process...")

    old_server.terminate()

    try:
        old_server.wait(timeout=10)
        print("Previous Streamlit process stopped.")
    except subprocess.TimeoutExpired:
        raise RuntimeError(
            "Previous server did not stop gracefully. "
            "No replacement server was launched."
        )
else:
    print("\nNo existing notebook-managed server running.")

# ------------------------------------------------------------
# 2. Prepare logging
# ------------------------------------------------------------

log_path = PROJECT_DIR / "streamlit_v2_1_1.log"

streamlit_log_handle = open(
    log_path,
    "a",
    buffering=1,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 3. Launch with Colab-compatible security configuration
# ------------------------------------------------------------

command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "app.py",
    "--server.port=8501",
    "--server.address=0.0.0.0",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false",
    "--browser.gatherUsageStats=false",
]

print("\nStarting Streamlit with Colab proxy compatibility...")

server = subprocess.Popen(
    command,
    cwd=PROJECT_DIR,
    stdout=streamlit_log_handle,
    stderr=subprocess.STDOUT
)

# ------------------------------------------------------------
# 4. Check startup
# ------------------------------------------------------------

time.sleep(5)

if server.poll() is not None:
    print("\nStreamlit startup failed.")

    print(
        log_path.read_text(
            encoding="utf-8",
            errors="replace"
        )[-8000:]
    )

    raise RuntimeError("Streamlit exited during startup.")

print("\nStreamlit process: RUNNING")

# ------------------------------------------------------------
# 5. Verify health endpoint
# ------------------------------------------------------------

try:
    with urllib.request.urlopen(
        f"http://127.0.0.1:{PORT}/_stcore/health",
        timeout=10
    ) as response:

        health = response.read().decode()

        print("Health status:", response.status)
        print("Health response:", health)

        if response.status != 200 or health.strip() != "ok":
            raise RuntimeError("Unexpected Streamlit health response.")

except Exception as error:
    raise RuntimeError(
        f"Streamlit health check failed: {error}"
    )

print("\n" + "=" * 60)
print("STREAMLIT REPAIR COMPLETED")
print("=" * 60)

# ------------------------------------------------------------
# 6. Open through Google Colab proxy
# ------------------------------------------------------------

output.serve_kernel_port_as_window(PORT)


MYR SUPPLYLENS — STREAMLIT REPAIR

Stopping existing Streamlit process...
Previous Streamlit process stopped.

Starting Streamlit with Colab proxy compatibility...

Streamlit process: RUNNING
Health status: 200
Health response: ok

STREAMLIT REPAIR COMPLETED
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [16]:

# ============================================================
# MYR SUPPLYLENS — STREAMLIT DIAGNOSTIC
# Read-only. Does not restart or terminate the runtime.
# ============================================================

from pathlib import Path
import subprocess
import socket
import urllib.request
import urllib.error

PROJECT_DIR = Path(PROJECT_DIR)
PORT = 8501

print("=" * 65)
print("MYR SUPPLYLENS — STREAMLIT DIAGNOSTICS")
print("=" * 65)

# 1. Check Streamlit process
print("\n[1] STREAMLIT PROCESS")

if "server" in globals():
    print("Original server process:", server.poll())
    print("None means process is still running.")
else:
    print("Original server variable not found.")

processes = subprocess.run(
    ["bash", "-lc",
     "ps -eo pid,ppid,stat,etime,args | grep -E '[s]treamlit run'"],
    capture_output=True,
    text=True
)

print(processes.stdout or "No Streamlit process found.")

# 2. Check port
print("\n[2] PORT 8501")

try:
    with socket.create_connection(
        ("127.0.0.1", PORT), timeout=3
    ):
        print("Port 8501: OPEN")
except OSError as e:
    print("Port 8501: CLOSED")
    print("Details:", e)

# 3. Check HTTP endpoints
print("\n[3] STREAMLIT HTTP HEALTH")

for endpoint in [
    "/_stcore/health",
    "/"
]:
    url = f"http://127.0.0.1:{PORT}{endpoint}"

    try:
        with urllib.request.urlopen(url, timeout=8) as response:
            print(
                endpoint,
                "HTTP", response.status,
                "Response:", response.read(150).decode(
                    "utf-8", errors="replace"
                )
            )
    except Exception as e:
        print(endpoint, "FAILED:", repr(e))

# 4. Read actual Streamlit logs
print("\n[4] STREAMLIT LOG")

log_path = PROJECT_DIR / "streamlit_v2_1_1.log"

print("Log path:", log_path)
print("Log exists:", log_path.exists())

if log_path.exists():
    log_text = log_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    print("\nLATEST LOG OUTPUT:")
    print("-" * 65)
    print(log_text[-12000:])
    print("-" * 65)

print("\nDIAGNOSTIC COMPLETED")


MYR SUPPLYLENS — STREAMLIT DIAGNOSTICS

[1] STREAMLIT PROCESS
Original server process: None
None means process is still running.
  11072    1013 Sl         05:25 /usr/bin/python3 -m streamlit run app.py --server.port 8501 --server.address 0.0.0.0


[2] PORT 8501
Port 8501: OPEN

[3] STREAMLIT HTTP HEALTH
/_stcore/health HTTP 200 Response: ok
/ HTTP 200 Response: <!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may n

[4] STREAMLIT LOG
Log path: /content/drive/MyDrive/MYR_SupplyLens/streamlit_v2_1_1.log
Log exists: True

LATEST LOG OUTPUT:
-----------------------------------------------------------------


2026-11-14 08:56:04.189 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.85.148:8501

2026-11-14 08:57:05.370 Rejecting WebSocket connection

## V2.1.1 safety notes

- This notebook contains no automatic restart, kernel shutdown or process-kill code.
- Do not run `runtime.restart_runtime()`, `ip.kernel.do_shutdown()`, `os.kill()` or `SIGKILL` manually.
- If a stale-import error occurs after installing packages, use Colab's UI: **Runtime → Restart session**, then rerun from Cell 1.
- The source patch backs up changed source files and does not delete or overwrite `data/` or `artifacts/`.
- V2.1 model artifacts remain `_v2_1_` and are loaded ahead of V2/V1 artifacts.
- Forecasts and empirical calibration ranges are decision-support estimates, not guaranteed outcomes or financial advice.